[Énoncés : Série 3 — Deux pointeurs et fenêtre glissante](../serie3_deux_pointeurs_fenetre.ipynb)

# Corrigé — Série 3 — Deux pointeurs et fenêtre glissante

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 13 — Écart minimal entre deux listes triées

**Idée** : un pointeur par liste. À chaque tour, on compare `a[i]` et `b[j]`, puis on fait avancer celui qui pointe la **plus petite** valeur : c'est le seul mouvement qui peut réduire l'écart.

**Complexité** : O(len(a) + len(b)). La double boucle serait en O(len(a) × len(b)).

In [ ]:
def ecart_minimal_tries(a: list[int], b: list[int]) -> int:
    i, j = 0, 0
    meilleur = abs(a[0] - b[0])
    while i < len(a) and j < len(b):
        meilleur = min(meilleur, abs(a[i] - b[j]))
        if a[i] < b[j]:
            i += 1      # a[i] est le plus petit : seule une valeur de a plus grande peut se rapprocher de b[j]
        else:
            j += 1
    return meilleur


verifier(ecart_minimal_tries, [
    (([1, 5, 11], [4, 9, 15]), 1),
    (([1, 2, 3], [10, 20]), 7),
    (([5], [5]), 0),
    (([-10, 0, 10], [3]), 3),
    (([1, 100], [50, 51, 99]), 1),
    (([10, 20, 30], [1, 2, 3]), 7),
])

In [ ]:
# Test de vitesse
chrono(ecart_minimal_tries, list(range(0, 20_000, 2)), list(range(1, 20_000, 2)), attendu=1)

## Exercice 14 — Inverser uniquement les voyelles

**Idée** : deux pointeurs aux deux bouts. Chacun avance jusqu'à la prochaine voyelle ; quand les deux sont sur une voyelle, on les échange. Les chaînes Python n'étant pas modifiables, on travaille sur une liste de caractères.

**Complexité** : O(n).

In [ ]:
def inverser_voyelles(texte: str) -> str:
    voyelles = set("aeiouAEIOU")
    lettres = list(texte)              # une chaîne n'est pas modifiable : on passe par une liste
    gauche, droite = 0, len(lettres) - 1
    while gauche < droite:
        if lettres[gauche] not in voyelles:
            gauche += 1
        elif lettres[droite] not in voyelles:
            droite -= 1
        else:
            lettres[gauche], lettres[droite] = lettres[droite], lettres[gauche]
            gauche += 1
            droite -= 1
    return "".join(lettres)


verifier(inverser_voyelles, [
    (('bonjour',), 'bunjoor'),
    (('Ali',), 'ilA'),
    (('',), ''),
    (('rythme',), 'rythme'),   # une seule voyelle
    (('xyz',), 'xyz'),
    (('aeiou',), 'uoiea'),
    (('école',), 'écelo'),   # é n'est pas une voyelle ici
    (('Hello World',), 'Hollo Werld'),
])

## Exercice 15 — Combien de paires donnent la cible ?

**Idée** : les deux pointeurs du thème 3, avec une gestion des doublons. Quand la somme est bonne, on compte le bloc de `g` copies à gauche et le bloc de `d` copies à droite : `g × d` paires. Cas particulier : si les deux valeurs sont égales, tout le segment a la même valeur et on choisit 2 éléments parmi `m`, soit `m × (m − 1) / 2`.

Autre solution O(n), plus simple : le dict compteur de `nb_paires` (thème 6), qui n'a même pas besoin que la liste soit triée.

**Complexité** : O(n).

In [ ]:
def nombre_paires_cible_trie(nombres: list[int], cible: int) -> int:
    gauche, droite = 0, len(nombres) - 1
    total = 0
    while gauche < droite:
        somme = nombres[gauche] + nombres[droite]
        if somme < cible:
            gauche += 1
        elif somme > cible:
            droite -= 1
        elif nombres[gauche] == nombres[droite]:
            # tout ce qui est entre gauche et droite a la même valeur : on choisit 2 éléments parmi m
            m = droite - gauche + 1
            total += m * (m - 1) // 2
            break
        else:
            g = 1                                   # nombre de copies de la valeur de gauche
            while nombres[gauche + g] == nombres[gauche]:
                g += 1
            d = 1                                   # nombre de copies de la valeur de droite
            while nombres[droite - d] == nombres[droite]:
                d += 1
            total += g * d                          # chaque copie de gauche avec chaque copie de droite
            gauche += g
            droite -= d
    return total


verifier(nombre_paires_cible_trie, [
    (([1, 1, 2, 3, 3, 4], 4), 4),
    (([2, 2, 2], 4), 3),   # trois 2 : 3 paires d'indices
    (([], 5), 0),
    (([1, 2, 3], 10), 0),
    (([1, 2, 3, 4, 5], 6), 2),
    (([0, 0, 0, 0], 0), 6),
    (([-2, -1, 0, 1, 2], 0), 2),
])

In [ ]:
# Test de vitesse
chrono(nombre_paires_cible_trie, [i // 150 for i in range(15_000)], 99, attendu=1_125_000)

## Exercice 16 — Première fenêtre avec une moyenne suffisante

**Idée** : fenêtre glissante de taille fixe (thème 3, partie 1). On met à jour la somme en O(1) à chaque décalage, et on compare la somme à `seuil × k` plutôt que de calculer une moyenne à virgule.

**Complexité** : O(n). Recalculer `sum(nombres[i:i+k])` pour chaque début coûterait O(n × k).

In [ ]:
def premiere_fenetre_moyenne(nombres: list[int], k: int, seuil: int) -> int:
    if k > len(nombres):
        return -1
    objectif = seuil * k                    # moyenne >= seuil  <=>  somme >= seuil × k
    somme = sum(nombres[:k])                # la première fenêtre
    if somme >= objectif:
        return 0
    for fin in range(k, len(nombres)):
        somme += nombres[fin] - nombres[fin - k]   # un élément entre à droite, un sort à gauche
        if somme >= objectif:
            return fin - k + 1                     # indice de début de cette fenêtre
    return -1


verifier(premiere_fenetre_moyenne, [
    (([2, 3, 5, 1, 4], 2, 4), 1),
    (([1, 1, 1], 2, 5), -1),
    (([5], 1, 5), 0),   # moyenne égale au seuil : ça compte
    (([1, 2], 3, 0), -1),   # moins de k éléments
    (([1, 2, 3, 4, 10], 3, 5), 2),
    (([0, 0, 9, 0], 1, 9), 2),
    (([-1, -1, -1], 3, -1), 0),
])

In [ ]:
# Test de vitesse
chrono(premiere_fenetre_moyenne, [0] * 99_999 + [1_000_000], 10_000, 100, attendu=90_000)

## Exercice 17 — Plus courte fenêtre avec assez de voyelles

**Idée** : la chenille du thème 3, « à l'envers ». La tête avance tant qu'il manque des voyelles ; **dès que** la fenêtre est valide, on note sa longueur et on avance la queue pour essayer une fenêtre plus courte. Variante : noter les positions des voyelles ; la réponse est le minimum de `pos[i + k - 1] - pos[i] + 1`.

**Complexité** : O(n) (chaque caractère entre et sort au plus une fois).

In [ ]:
def plus_courte_voyelles(texte: str, k: int) -> int:
    voyelles = set("aeiou")
    debut = 0
    nb = 0                          # nombre de voyelles dans texte[debut : fin + 1]
    meilleur = 0                    # 0 = pas encore de solution
    for fin, c in enumerate(texte):
        if c in voyelles:
            nb += 1
        while nb >= k:              # fenêtre valide : on essaie de la raccourcir par la gauche
            longueur = fin - debut + 1
            if meilleur == 0 or longueur < meilleur:
                meilleur = longueur
            if texte[debut] in voyelles:
                nb -= 1
            debut += 1
    return meilleur


verifier(plus_courte_voyelles, [
    (('baceido', 3), 4),
    (('xyz', 1), 0),   # aucune voyelle
    (('a', 1), 1),
    (('aeiou', 5), 5),
    (('abbbbbe', 2), 7),
    (('bcaxxexi', 2), 3),
    (('aaa', 4), 0),   # pas assez de voyelles
])

## Exercice 18 — Palindrome avec un caractère à retirer

**Idée** : deux pointeurs qui se rapprochent. Tant que les caractères sont égaux, on avance. Au **premier** désaccord, il n'y a que deux possibilités : retirer le caractère de gauche ou celui de droite ; on vérifie le reste dans les deux cas.

**Complexité** : O(n). Essayer de retirer chaque caractère un par un coûterait O(n²).

In [ ]:
def palindrome_un_retrait(texte: str) -> bool:
    def est_palindrome(g, d):              # texte[g..d] est-il un palindrome ?
        while g < d:
            if texte[g] != texte[d]:
                return False
            g += 1
            d -= 1
        return True

    gauche, droite = 0, len(texte) - 1
    while gauche < droite:
        if texte[gauche] != texte[droite]:
            # premier désaccord : on a droit à UN retrait, à gauche ou à droite
            return est_palindrome(gauche + 1, droite) or est_palindrome(gauche, droite - 1)
        gauche += 1
        droite -= 1
    return True


verifier(palindrome_un_retrait, [
    (('abca',), True),
    (('abc',), False),
    (('',), True),
    (('a',), True),
    (('radar',), True),   # déjà un palindrome
    (('ab',), True),
    (('abcdba',), True),
    (('abccaa',), False),
    (('deeee',), True),
])

In [ ]:
# Test de vitesse
chrono(palindrome_un_retrait, "a" * 50_000 + "bcd" + "a" * 50_000, attendu=False)

---

[Énoncés de la série](../serie3_deux_pointeurs_fenetre.ipynb)